# Qwen3.5-9B LoRA Fine-Tuning — Computer Use

## Overview

This notebook fine-tunes a **LoRA adapter** on top of Qwen3.5-9B using a synthetically generated computer-use dataset. The result is a tiny adapter file (~110 MB) that, when attached to the base model, gives it native tool-use and computer-control capabilities.

## Training strategy

- **LoRA (Low-Rank Adaptation)** — only rank-16 delta matrices are trained; base weights stay frozen.
- **Unsloth** — CUDA-kernel-patched model loader with fused RoPE, FlashAttention, and smart gradient checkpointing to minimise VRAM.
- **bfloat16 throughout** — no 4-bit quantisation during training so the adapter can be merged cleanly into fp16 later.
- **Multi-turn `CompletionOnlyCollator`** — computes loss only on assistant turns, maximising signal from multi-step tool-use conversations.

## Outputs

Adapter checkpoint (`OUTPUT_DIR`) containing:
- `adapter_config.json` — LoRA hyperparameters + base model reference
- `adapter_model.safetensors` — trained LoRA weight matrices
- Tokenizer files

> **After training**, use the `[Checkpoint_Resume]` notebook to merge the adapter into the base model and export to GGUF.

## Cell 1 — Imports

All required libraries. `time` is used to measure training duration; `SFTTrainer` / `SFTConfig` are TRL's supervised fine-tuning wrappers that integrate directly with Unsloth.

In [ ]:
!pip install -r requirements.txt
!pip install -U ipywidgets jupyterlab_widgets widgetsnbextension
!pip install --upgrade --force-reinstall typing_extensions

In [ ]:
from unsloth import FastLanguageModel
import os
import time
import json
import shutil
import torch
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig
from peft import PeftModel

print("Torch:", torch.__version__, "CUDA available:", torch.cuda.is_available())
print("Unsloth + TRL import OK 🦥")

## Cell 2 — Configuration

**Edit these variables before running.** All subsequent cells derive from them.

| Variable | Description |
|---|---|
| `MODEL_NAME` | HuggingFace model ID for the Qwen3.5-9B base (loaded from Hub or local cache) |
| `DATASET_PATH` | Path to the JSONL file containing synthetic computer-use conversations |
| `OUTPUT_DIR` | Directory where adapter checkpoints will be saved during and after training |
| `MAX_SEQ_LENGTH` | Maximum token length per example — 4096 is a good balance for tool-use conversations |

In [ ]:
# ── Model ────────────────────────────────────────────────────────────────
# Priority: Env var (OI_MCP_PATH) > Current directory discovery
import os

MODEL_NAME            = "Jackrong/Qwen3.5-27B-Claude-4.6-Opus-Reasoning-Distilled"
BASE_DIR              = "models/base"
DATASET_PATH          = "synthetic_dataset_final.jsonl"
ADAPTER_CHECKPOINT    = "models/adapter-checkpoint"
OUTPUT_DIR            = "models/qwen-tool-use-lora-adapter-qwen-27b-gguf"

os.environ["HF_HOME"] = BASE_DIR
os.environ["TRANSFORMERS_CACHE"] = BASE_DIR  # where transformers stores models

os.makedirs(BASE_DIR, exist_ok=True)

# ── Sequence length ────────────────────────────────────────────────────────
MAX_SEQ_LENGTH = 2048

print(f"Base dir   : {BASE_DIR}")
print(f"Model      : {MODEL_NAME}")
print(f"Dataset    : {DATASET_PATH}")
print(f"Adapter Chekpoint dir    : {ADAPTER_CHECKPOINT}")
print(f"Output dir : {OUTPUT_DIR}")

## Cell 3 — Load Model & Tokenizer

Load Qwen3.5-9B via Unsloth's optimised loader. Unsloth automatically applies:
- Fused RoPE embeddings (faster + less memory)
- FlashAttention-2 (if supported)
- Smart gradient checkpointing ready for the LoRA step

`load_in_4bit=True` is used during training to keep VRAM within 16 GB; the 4-bit quantisation is applied only to the frozen base weights and does **not** affect training precision (LoRA layers train in bf16). `trust_remote_code=True` is required for Qwen's custom tokenizer code.

In [ ]:
# Load the base model + tokenizer using Unsloth's kernel-patched loader.
# Unsloth automatically applies fused RoPE, FlashAttention, and other speed-ups.
model, tokenizer = FastLanguageModel.from_pretrained(
    cache_dir         = BASE_DIR,
    model_name        = MODEL_NAME,
    max_seq_length    = MAX_SEQ_LENGTH,   # 2048 is safe; 4096 also OK on 80GB
    dtype             = torch.bfloat16,
    load_in_4bit      = True,             # keep this; QLoRA on 27B
    trust_remote_code = True,
)
print(f"Model loaded: {MODEL_NAME}")

## Cell 4 — Dataset Preparation

Load the JSONL dataset and apply Qwen's native chat template to convert each `messages` list into a single formatted string.

- `apply_chat_template` renders system/user/assistant turns into `<|im_start|>...<|im_end|>` format that matches Qwen's pre-training.
- `add_generation_prompt=False` ensures the trailing `<|im_start|>assistant\n` prompt is **not** added — we want the full turn including the assistant response so the model learns to produce it.
- The resulting `text` field is what the tokenizer will later consume.

In [ ]:
import json
from datasets import Dataset

print(f"Loading and sanitizing dataset from {DATASET_PATH}...")

cleaned_data = []
dropped_lines = 0

# 1. Read and forcefully sanitize the dataset using pure Python
with open(DATASET_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
            
        try:
            row = json.loads(line)
            if "messages" not in row:
                continue
            
            clean_messages = []
            for msg in row["messages"]:
                if not isinstance(msg, dict):
                    continue
                
                # Core fields (Safe fallback for None types)
                clean_msg = {
                    "role": str(msg.get("role", "user")),
                    "content": msg.get("content", "") if msg.get("content") is not None else ""
                }
                
                # If it's a tool response, it needs the ID
                if "tool_call_id" in msg:
                    clean_msg["tool_call_id"] = str(msg["tool_call_id"])
                    
                # Forcefully normalize tool calls into the strict OpenAI/Qwen standard format
                if "tool_calls" in msg and isinstance(msg["tool_calls"], list):
                    clean_tool_calls = []
                    for tc in msg["tool_calls"]:
                        if not isinstance(tc, dict):
                            continue
                        
                        func_name = ""
                        func_args = "{}"
                        
                        if "function" in tc and isinstance(tc["function"], dict):
                            func_name = tc["function"].get("name", "")
                            args_raw = tc["function"].get("arguments", {})
                        else:
                            func_name = tc.get("name", "")
                            args_raw = tc.get("arguments", {})
                            
                        func_args = args_raw if isinstance(args_raw, str) else json.dumps(args_raw)
                        
                        clean_tool_calls.append({
                            "type": "function",
                            "function": {
                                "name": str(func_name),
                                "arguments": str(func_args)
                            }
                        })
                        
                    if clean_tool_calls:
                        clean_msg["tool_calls"] = clean_tool_calls
                        
                clean_messages.append(clean_msg)
            
            cleaned_data.append({"messages": clean_messages})
            
        except Exception:
            dropped_lines += 1

# 2. Build the Hugging Face dataset
dataset = Dataset.from_list(cleaned_data)
print(f"Dataset size: {len(dataset)} examples. Dropped {dropped_lines} completely unreadable lines.")

# 3. Apply the Chat Template
def format_chat_template(example):
    example["text"] = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return example

dataset = dataset.map(format_chat_template)
dataset = dataset.filter(lambda x: x.get("text") is not None and len(x["text"]) > 10)

# ==========================================
# NEW: 4. Tokenize the text into input_ids
# ==========================================
# We need to extract the inner tokenizer to bypass the Qwen3VLProcessor wrapper
inner_tokenizer = tokenizer.tokenizer if hasattr(tokenizer, "tokenizer") else tokenizer

def tokenize_function(example):
    # This converts the raw string into numerical IDs and an attention mask
    return inner_tokenizer(
        example["text"],
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding=False, # We let the collator handle dynamic padding later
    )

print("Tokenizing dataset...")
dataset = dataset.map(
    tokenize_function,
    remove_columns=["messages", "text"], # Remove the raw text to save RAM
    desc="Tokenizing",
)

print(f"Tokenization complete. Ready for training!")

## Cell 5 — LoRA Adapter Configuration

Wrap the frozen base model with trainable LoRA rank-16 matrices using Unsloth's `get_peft_model`.

**Why these target modules?**
- `q_proj`, `k_proj`, `v_proj`, `o_proj` — attention projections; fine-tuning these teaches the model *what to attend to* in tool-use contexts.
- `gate_proj`, `up_proj`, `down_proj` — FFN projections; fine-tuning these teaches the model *what to say* (the actual tool-call JSON, reasoning, etc.).

**LoRA hyperparameters:**

| Param | Value | Reason |
|---|---|---|
| `r` | 16 | Good capacity for a domain-specific task; higher r = more trainable params |
| `lora_alpha` | 16 | `alpha/r = 1.0` effective scale — no extra amplification |
| `lora_dropout` | 0 | Disabled; Unsloth has special optimisations for `dropout=0` |
| `bias` | none | Disabled; Unsloth has special optimisations for `bias=none` |

In [ ]:
print("Attaching LoRA adapter to the base model...")

model = FastLanguageModel.get_peft_model(
    model,
    r                          = 32,                 
    target_modules             = [                   
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_alpha                 = 32,                 
    lora_dropout               = 0,                  
    bias                       = "none",             
    use_gradient_checkpointing = True,  # FIX: Reverted to standard PyTorch checkpointing
    random_state               = 3407,               
)
model.print_trainable_parameters()

## Cell 6 — Trainer Setup (Collator + Tokenization + SFTTrainer)

This is the core training setup cell. It handles three things:

### 6a. EOS/PAD token synchronisation
Qwen's tokenizer wraps an inner tokenizer inside a `Processor`. Both must have matching EOS and PAD tokens to avoid training on incorrect end-of-sequence boundaries. `<|im_end|>` is Qwen's native turn delimiter and is the correct EOS for this format.

### 6b. `CompletionOnlyCollator`
A custom data collator that:
- Sets all token labels to `-100` (masked, no gradient) by default.
- Scans each sequence for `<|im_start|>assistant\n` markers.
- Unmasks **every** assistant turn found — not just the first one.

This is critical for multi-step tool-use conversations where the model must produce multiple assistant turns (reasoning → tool call → observe result → final answer). A single-turn collator would lose 80%+ of training signal.

### 6c. Pre-tokenization
Tokenizes the full dataset in the main process before training. This avoids a known pickling crash in DataLoader worker processes when using Unsloth's wrapped tokenizer.

### 6d. `SFTTrainer` + `SFTConfig`
Key hyperparameters:

| Param | Value | Reason |
|---|---|---|
| `per_device_train_batch_size` | 2 | Fits in 16 GB VRAM with bf16 + gradient checkpointing |
| `gradient_accumulation_steps` | 4 | Effective batch = 2×4 = 8 — stable gradients |
| `warmup_steps` | 50 | ~5% of 900 steps — prevents early instability |
| `max_steps` | 900 | ~5 epochs on the dataset |
| `learning_rate` | 2e-4 | Standard LoRA LR; cosine decay from here |
| `save_steps` | 100 | Checkpoint every 100 steps |
| `lr_scheduler_type` | cosine | Smooth LR decay towards end of training |

In [ ]:
# ---------------------------------------------------------------------------
# TRAINER SETUP v3 — Multi-turn collator + VRAM Optimized
# ---------------------------------------------------------------------------
import os
from trl import SFTTrainer, SFTConfig

os.environ["TOKENIZERS_PARALLELISM"] = "false"

# 1. Find correct EOS from inner tokenizer
inner_tokenizer = tokenizer.tokenizer if hasattr(tokenizer, "tokenizer") else tokenizer
real_eos_token = "<|im_end|>"
for token in ["<|im_end|>", "<|endoftext|>", "</s>"]:
    try:
        if token in inner_tokenizer.get_vocab():
            real_eos_token = token
            break
    except Exception:
        continue
real_eos_id = inner_tokenizer.convert_tokens_to_ids(real_eos_token)
print(f"EOS token: '{real_eos_token}' (id={real_eos_id})")

# 2. Forced Synchronization
for obj in [tokenizer, inner_tokenizer, model.config]:
    setattr(obj, "eos_token",    real_eos_token)
    setattr(obj, "eos_token_id", real_eos_id)
    setattr(obj, "pad_token",    real_eos_token)
    setattr(obj, "pad_token_id", real_eos_id)
if hasattr(inner_tokenizer, "add_special_tokens"):
    inner_tokenizer.add_special_tokens({"eos_token": real_eos_token, "pad_token": real_eos_token})

# 3. MULTI-TURN CompletionOnlyCollator
# FIX applied here: Using inner_tokenizer instead of tokenizer
IM_START_ID = inner_tokenizer.encode("<|im_start|>", add_special_tokens=False)[0] 

class CompletionOnlyCollator:
    """Masks all prompt/system/user tokens. Trains on ALL assistant turns."""
    def __init__(self, response_template, tokenizer):
        self.tokenizer = tokenizer
        self.response_template_ids = tokenizer.encode(
            response_template, add_special_tokens=False
        )

    def __call__(self, features):
        batch = self.tokenizer.pad(features, return_tensors="pt", padding=True)
        labels = batch["input_ids"].clone()
        labels[:] = -100  # mask everything by default

        tpl = self.response_template_ids
        n = len(tpl)

        for i in range(len(labels)):
            ids = batch["input_ids"][i].tolist()
            j = 0
            while j <= len(ids) - n:
                if ids[j:j + n] == tpl:
                    response_start = j + n
                    response_end = len(ids)
                    for k in range(response_start, len(ids)):
                        if ids[k] == IM_START_ID:
                            response_end = k
                            break
                    labels[i][response_start:response_end] = \
                        batch["input_ids"][i][response_start:response_end]
                    j = response_end
                else:
                    j = j + 1

        batch["labels"] = labels
        return batch

response_template = "<|im_start|>assistant\n"
collator = CompletionOnlyCollator(response_template, inner_tokenizer)

# 4. Training Config - A100 MAXIMUM OVERDRIVE
training_args = SFTConfig(
    output_dir=ADAPTER_CHECKPOINT,
    per_device_train_batch_size=16,  # Doubled to eat up the remaining 32GB VRAM
    gradient_accumulation_steps=1,   # Dropped to 1 for maximum parallel throughput
    dataloader_num_workers=4,        # Keeps the GPU fed constantly
    learning_rate=5e-5,
    logging_steps=10,
    max_steps=2000,                  # CUT IN HALF: 5 epochs is plenty for tool-use LoRA
    warmup_steps=100,                # Adjusted for the shorter step count
    save_steps=500,
    optim="adamw_8bit",              
    fp16=False,
    bf16=True,
    report_to="none",
    max_length=MAX_SEQ_LENGTH,
    dataset_num_proc=4,              
    dataset_kwargs={"skip_prepare_dataset": True},
    remove_unused_columns=False,
)
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,           # Passing the clean RAM dataset we built in Cell 4
    processing_class=inner_tokenizer,
    args=training_args,
    data_collator=collator,
)

print(f"Trainer ready. Batches optimized for local VRAM constraints.")
print(f"Trainer EOS Token determined as: {real_eos_token} (ID: {real_eos_id})")

## Cell 7 — Training Execution + Metrics + Adapter Save

This cell fires the actual training run and handles three phases:

### 7a. Pre-flight GPU snapshot
Captures current VRAM reservation and total GPU memory before training begins. Printed at the end for comparison.

### 7b. `trainer.train()`
Launches the SFTTrainer loop. Unsloth's `ACCELERATE_BYPASS_DEVICE_MAP=true` environment flag is required to avoid a device-map conflict when using its custom model wrapper.

### 7c. Post-training metrics
After training completes:
- **Total Training Time** — wall-clock minutes for the full run
- **Peak VRAM Usage** — maximum GPU memory reserved during training (GB)
- **Steps/Sec** — throughput sampled from `train_samples_per_second`

### 7d. Adapter save
Only the **LoRA delta weights** are saved — not the full 9B base model. This produces a tiny `~110 MB` adapter directory containing:
- `adapter_config.json` — LoRA rank, alpha, target modules, base model reference
- `adapter_model.safetensors` — the trained rank-16 weight matrices
- Tokenizer files (for standalone inference)

> **Note:** To use the adapter you must load the base model first and attach the adapter via `PeftModel.from_pretrained()` or Unsloth's `from_pretrained()` with `adapter_name`.

In [ ]:
# ---------------------------------------------------------------------------
# TRAINING EXECUTON
# ---------------------------------------------------------------------------
print("Starting ultra-fast Unsloth training...")
start_time = time.time()
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

os.environ["ACCELERATE_BYPASS_DEVICE_MAP"] = "true"
trainer_stats = trainer.train()

end_time = time.time()
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
print(f"\n--- TRAINING METRICS ---")
print(f"Total Training Time: {(end_time - start_time)/60:.2f} minutes")
print(f"Peak VRAM Usage:     {used_memory} GB")
print(f"Steps/Sec:           {trainer_stats.metrics['train_samples_per_second']:.2f}")

# Save ONLY the adapters
print(f"Finished! Saving LoRA adapters to {ADAPTER_CHECKPOINT}")
model.save_pretrained(ADAPTER_CHECKPOINT) # Local saving
tokenizer.save_pretrained(ADAPTER_CHECKPOINT)
print(f"✅ Adapter successfully saved!")

## Cell 8 — Export to GGUF (Optional)

Converts the merged adapter + base model into a standalone **GGUF** file for hardware-agnostic local inference. GGUF is the native format for `llama.cpp` and compatible runtimes (Ollama, LM Studio, koboldcpp, etc.).

### What `save_pretrained_gguf` does
1. **Merges** the LoRA adapter back into the base model weights (fp16)
2. **Quantises** the merged model using the specified method
3. **Writes** a single `.gguf` file to the `"model"` directory

### Quantisation method: `q4_k_m`
| Method | Bits | Size (9B) | Quality vs Size |
|--------|------|-----------|------------------|
| `q4_k_m` | 4-bit | ~5.7 GB | Best balance — recommended for most hardware |
| `q8_0` | 8-bit | ~9.4 GB | Near lossless, requires more RAM |
| `f16` | 16-bit | ~17 GB | Full precision, for fine-tuning or re-export |

> **Prerequisite:** This cell requires that the adapter was saved in Cell 7 and that the base model weights are accessible on disk. The merge step loads the full fp16 base model (~18 GB) into RAM temporarily.

> **Tip:** Skip this cell if you only need the raw adapter for server-side inference with VLLM or TGI — those load the adapter directly without needing a GGUF merge.

In [1]:
# Run this IMMEDIATELY after restarting the kernel!
import os
import gc
import torch
import types
import unsloth.save
from unsloth import FastLanguageModel
from peft import PeftModel

# Configuration variables (make sure these match your paths)
MAX_SEQ_LENGTH = 2048
ADAPTER_CHECKPOINT = "models/adapter-checkpoint"
OUTPUT_DIR = "models/adapter-checkpoint" 

print("Loading base model into clean VRAM...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name        = "models/base/models--Jackrong--Qwen3.5-27B-Claude-4.6-Opus-Reasoning-Distilled/snapshots/59f57e471e041fe27ee3f98dba2ec02a50817afc",
    max_seq_length    = MAX_SEQ_LENGTH,
    dtype             = torch.bfloat16,
    load_in_4bit      = True,          
    trust_remote_code = True,
)

print("Attaching LoRA adapter...")
model = PeftModel.from_pretrained(model, ADAPTER_CHECKPOINT + "/checkpoint-1000")

# Glue Unsloth's GGUF methods back onto the PEFT model
model.save_pretrained_gguf = types.MethodType(unsloth.save.unsloth_save_pretrained_gguf, model)

# Apply the Unsloth Tokenizer Patch
def bypass_bos_check(tok):
    return False, getattr(tok, "chat_template", None)
unsloth.save.fix_tokenizer_bos_token = bypass_bos_check

# Export the highly compressed quantizations
quants = ["q4_k_m", "q5_k_m"]  
GGUF_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "final-qwen-gguf-export")

print(f"Exporting quantizations: {quants}")
model.save_pretrained_gguf(
    GGUF_OUTPUT_DIR,
    tokenizer,             
    quantization_method  = quants, 
    maximum_memory_usage = 0.9,   
)

print(f"✅ Export complete! Check the {GGUF_OUTPUT_DIR} folder.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading base model into clean VRAM...
Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2026.3.4: Fast Qwen3_5 patching. Transformers: 5.2.0.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.252 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Unsloth: Qwen3_5 does not support SDPA - switching to fast eager.


Loading weights:   0%|          | 0/1184 [00:00<?, ?it/s]

models/base/models--Jackrong--Qwen3.5-27B-Claude-4.6-Opus-Reasoning-Distilled/snapshots/59f57e471e041fe27ee3f98dba2ec02a50817afc does not have a padding token! Will use pad_token = <|endoftext|>.
Attaching LoRA adapter...
Exporting quantizations: ['q4_k_m', 'q5_k_m']
Unsloth: Merging model weights to 16-bit format...
Detected local model directory: /workspace/models/base/models--Jackrong--Qwen3.5-27B-Claude-4.6-Opus-Reasoning-Distilled/snapshots/59f57e471e041fe27ee3f98dba2ec02a50817afc
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


...

Unsloth: Merging weights into 16bit: 100%|██████████| 11/11 [04:45<00:00, 25.94s/it]


Unsloth: Merge process complete. Saved to `/workspace/models/adapter-checkpoint/final-qwen-gguf-export`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF bf16 might take 3 minutes.
\        /    [2] Converting GGUF bf16 to ['q4_k_m', 'q5_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: llama.cpp found in the system. Skipping installation.
Unsloth: Preparing converter script...


[unsloth_zoo.llama_cpp|WARNING]Unsloth: Qwen2MoE num_experts patch target not found.


Unsloth: [1] Converting model into bf16 GGUF format.
This might take 3 minutes...


RuntimeError: Unsloth: GGUF conversion failed: Unsloth: Failed to convert text model to GGUF with command `/usr/bin/python /root/.unsloth/llama.cpp/unsloth_convert_hf_to_gguf.py --outfile 59f57e471e041fe27ee3f98dba2ec02a50817afc.BF16.gguf --outtype bf16 --split-max-size 50G models/adapter-checkpoint/final-qwen-gguf-export`: Command '['/usr/bin/python', '/root/.unsloth/llama.cpp/unsloth_convert_hf_to_gguf.py', '--outfile', '59f57e471e041fe27ee3f98dba2ec02a50817afc.BF16.gguf', '--outtype', 'bf16', '--split-max-size', '50G', 'models/adapter-checkpoint/final-qwen-gguf-export']' returned non-zero exit status 1.

In [ ]:
import os

# ==========================================
# 1. RE-APPLY THE PATCH (Defeat the auto-updater)
# ==========================================
script_path = '/root/.unsloth/llama.cpp/unsloth_convert_hf_to_gguf.py'

with open(script_path, 'r') as f:
    code = f.read()

target = 'raise NotImplementedError("BPE pre-tokenizer was not recognized - update get_vocab_base_pre()")'
replacement = 'return "qwen2"  # PATCHED'

if target in code:
    code = code.replace(target, replacement)
    with open(script_path, 'w') as f:
        f.write(code)
    print("✅ llama.cpp patched successfully!")
else:
    print("✅ Patch already present!")

# ==========================================
# 2. RUN LLAMA.CPP DIRECTLY IN BASH
# ==========================================
import subprocess

# Clear old failed files to ensure maximum disk space
os.system("rm -f Qwen-27B-ToolUse*.gguf")
os.system("rm -f 59f57e471e041fe27ee3f98dba2ec02a50817afc*.gguf")

bash_commands = """
echo "--- STEP 1: Converting to 8-bit GGUF (Saving disk space!) ---"
/usr/bin/python /root/.unsloth/llama.cpp/unsloth_convert_hf_to_gguf.py \
    models/adapter-checkpoint/final-qwen-gguf-export \
    --outfile Qwen-27B-ToolUse.q8_0.gguf \
    --outtype q8_0

echo "--- STEP 2: Quantizing to final 4-bit GGUF ---"
/root/.unsloth/llama.cpp/llama-quantize \
    Qwen-27B-ToolUse.q8_0.gguf \
    Qwen-27B-ToolUse.q4_k_m.gguf \
    q4_k_m

echo "--- STEP 3: Cleaning up massive intermediate file ---"
rm Qwen-27B-ToolUse.q8_0.gguf

echo "✅ ALL DONE! Your Qwen-27B-ToolUse.q4_k_m.gguf is ready to download!"
"""

# Execute the bash commands and print output in real-time
process = subprocess.Popen(bash_commands, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in process.stdout:
    print(line, end="")